In [9]:
from openai import OpenAI
from openai import APIError, RateLimitError, APITimeoutError, APIConnectionError
import os
from dotenv import load_dotenv

# 加载环境变量 + 初始化客户端
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
base_url = "https://ark.cn-beijing.volces.com/api/v3"
client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    timeout=30.0
)

# ========== 工具函数：全部统一定义在最前面 ==========
def estimate_token(text: str) -> int:
    """简易估算中文文本的token数量"""
    char_count = len(text.strip())
    return int(char_count / 1.5)

def count_history_token(history: list) -> int:
    """估算整个history对话的总token数"""
    total = 0
    for msg in history:
        total += estimate_token(msg["content"])
    return total

def trim_history(history: list, max_round=4):
    """按对话轮数截断历史，固定保留system消息"""
    max_msg_count = 1 + max_round * 2
    while len(history) > max_msg_count:
        removed_user = history.pop(1)
        removed_assistant = history.pop(1)
        print(f"\n【历史截断】删除旧对话：user:{removed_user['content']}, ai:{removed_assistant['content']}")
    return history

def trim_history_by_token(history: list, max_token: int = 2000) -> list:
    """按token上限截断历史，固定保留system消息"""
    while count_history_token(history) > max_token and len(history) > 1:
        removed_user = history.pop(1)
        removed_assistant = history.pop(1)
        print(f"\n【历史截断】删除旧对话：user:{removed_user['content']}, ai:{removed_assistant['content']}")
    return history

# ========== 核心对话函数 ==========
def llm_chat(messages: list, model="ep-20260324112743-tglmk"):
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.3
        )
        choice = resp.choices[0]
        if choice.finish_reason == "length":
            return "【警告】输出达到最大长度限制，回答被截断\n" + choice.message.content
        if choice.finish_reason == "content_filter":
            return "【警告】内容命中安全过滤，已终止回答"
        return choice.message.content
    except RateLimitError:
        return "【异常】触发接口限流，请降低调用频率稍后重试"
    except APITimeoutError:
        return "【异常】请求超时，服务器响应时间过长"
    except APIConnectionError:
        return "【异常】网络连接失败，请检查网络"
    except APIError as e:
        return f"【接口错误】{e}"
    except Exception as e:
        return f"【未知异常】{e}"

# ========== 主程序 ==========
if __name__ == "__main__":
    history = [
        {"role": "system", "content": "你是AI技术助教，回答简洁易懂"}
    ]
    print("开始对话，输入quit退出聊天")

    while True:
        user_input = input("\n你：")
        if user_input.strip() == "quit":
            print("对话结束")
            break

        history.append({"role": "user", "content": user_input})
        ai_resp = llm_chat(history)

        # 空返回校验
        if ai_resp is None or len(ai_resp.strip()) == 0:
            ai_resp = "【模型返回内容为空】"
        else:
            history.append({"role": "assistant", "content": ai_resp})
            # 二选一：按轮数截断 / 按token截断
            trim_history(history, max_round=4)
            # trim_history_by_token(history, max_token=2000)

        total_token = count_history_token(history)
        print(f"AI：{ai_resp}")
        print(f"当前上下文预估token：{total_token}")


开始对话，输入quit退出聊天



你： nihao,woshidashabi


AI：请使用文明用语哦～如果有关于AI技术的问题，比如编程、算法、机器学习相关的内容，都可以随时问我，我会尽力帮你解答😊
当前上下文预估token：60



你： quit


对话结束
